# Classifier Two-Sample Test (C2ST)

## Imports

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

## Setup

In [ ]:
df = pl.read_csv("../../../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv")
neighbors = pl.read_csv("../../../dataset/ca_county_neighbors.csv").filter(pl.col("county_fips") < pl.col("neighbor_fips"))

FIPS_TO_NAME = {
    6001: "Alameda", 6003: "Alpine", 6005: "Amador", 6007: "Butte", 6009: "Calaveras",
    6011: "Colusa", 6013: "Contra Costa", 6015: "Del Norte", 6017: "El Dorado", 6019: "Fresno",
    6021: "Glenn", 6023: "Humboldt", 6025: "Imperial", 6027: "Inyo", 6029: "Kern",
    6031: "Kings", 6033: "Lake", 6035: "Lassen", 6037: "Los Angeles", 6039: "Madera",
    6041: "Marin", 6043: "Mariposa", 6045: "Mendocino", 6047: "Merced", 6049: "Modoc",
    6051: "Mono", 6053: "Monterey", 6055: "Napa", 6057: "Nevada", 6059: "Orange",
    6061: "Placer", 6063: "Plumas", 6065: "Riverside", 6067: "Sacramento", 6069: "San Benito",
    6071: "San Bernardino", 6073: "San Diego", 6075: "San Francisco", 6077: "San Joaquin",
    6079: "San Luis Obispo", 6081: "San Mateo", 6083: "Santa Barbara", 6085: "Santa Clara",
    6087: "Santa Cruz", 6089: "Shasta", 6091: "Sierra", 6093: "Siskiyou", 6095: "Solano",
    6097: "Sonoma", 6099: "Stanislaus", 6101: "Sutter", 6103: "Tehama", 6105: "Trinity",
    6107: "Tulare", 6109: "Tuolumne", 6111: "Ventura", 6113: "Yolo", 6115: "Yuba"
}

lc_types = df["lc_type"].unique().to_list()

df.shape, len(neighbors), len(lc_types)

## C2ST conditioned on land cover
Features used: `st_damcat`, `bldgtype`, `clr` (excluding `lc_type` since we're conditioning on it)

In [ ]:
FEATURES_NO_LC = ["st_damcat", "bldgtype", "clr"]
cat_features_no_lc = list(range(len(FEATURES_NO_LC)))

def c2st_with_importance(fips_a, fips_b, lc_type, n_splits=3):
    df_a = df.filter((pl.col("fips") == fips_a) & (pl.col("lc_type") == lc_type)).select(FEATURES_NO_LC)
    df_b = df.filter((pl.col("fips") == fips_b) & (pl.col("lc_type") == lc_type)).select(FEATURES_NO_LC)
    
    if len(df_a) < 50 or len(df_b) < 50:
        return None, len(df_a), len(df_b), None, None, None
    
    combined = pl.concat([
        df_a.with_columns(pl.lit(0).alias("label")),
        df_b.with_columns(pl.lit(1).alias("label"))
    ])
    
    X = combined.select(FEATURES_NO_LC).to_pandas()
    y = combined["label"].to_numpy()
    
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_idx, test_idx in cv.split(X, y):
        clf = CatBoostClassifier(iterations=100, depth=5, cat_features=cat_features_no_lc, verbose=False, random_state=42)
        clf.fit(X.iloc[train_idx], y[train_idx])
        scores.append(clf.score(X.iloc[test_idx], y[test_idx]))
    
    clf_final = CatBoostClassifier(iterations=100, depth=5, cat_features=cat_features_no_lc, verbose=False, random_state=42)
    clf_final.fit(X, y)
    imp = clf_final.feature_importances_
    
    return np.mean(scores), len(df_a), len(df_b), imp[0], imp[1], imp[2]

## Train all neighbor pairs × all land-cover types

In [ ]:
all_results = []

for lc in tqdm(lc_types, desc="Land cover types"):
    for row in neighbors.iter_rows(named=True):
        acc, n_a, n_b, imp_sd, imp_bt, imp_clr = c2st_with_importance(row["county_fips"], row["neighbor_fips"], lc)
        all_results.append({
            "fips_a": row["county_fips"],
            "fips_b": row["neighbor_fips"],
            "lc_type": lc,
            "accuracy": acc,
            "n_a": n_a,
            "n_b": n_b,
            "imp_st_damcat": imp_sd,
            "imp_bldgtype": imp_bt,
            "imp_clr": imp_clr
        })

all_results_df = pl.DataFrame(all_results)
all_results_df.shape

## Save results

In [ ]:
all_results_df.write_csv("c2st_results_all_lc.csv")

all_results_df.filter(pl.col("accuracy").is_not_null()).sort("accuracy", descending=True).head(10)

## Interpretation and handoff

The CSV retains one C2ST result per neighboring county pair and land-cover type; use its cross-validated accuracy and feature importances for downstream comparison.